# 01 · UCI HAR — Common Preprocessing Pipeline
**ICT-4442 · Comparative Analysis of Deep Learning Architectures for Smartphone HAR**

Run this notebook **once**. It reads the dataset from `My Drive/DL Proj/dataset/` and writes one shared, model-ready dataset to `My Drive/DL Proj/processed/` that all four models (MLP, 1D-CNN, LSTM/BiLSTM, Transformer) load. Nobody re-preprocesses, so every model trains and tests on exactly the same windows.

**Pipeline**

| Step | What happens | Fitted on |
|---|---|---|
| 1. Locate & validate | Finds the dataset in Drive, checks every file, row counts, label/subject ranges, alignment | — |
| 2. Clean | Unique feature names, NaN/inf repair, duplicate rows, constant features, cross-split duplicate windows | — |
| 3. Encode | Labels 1–6 → 0–5 in canonical UCI order | — |
| 4. Split | **Subject-independent**: official 9 test subjects; validation = ~4 whole training subjects, class-stratified | — |
| 5. Outlier clipping | Raw signals clipped at the 0.01 / 99.99 percentile per channel | train only |
| 6. Normalize | Tabular: StandardScaler. Sequences: per-channel z-score | train only |
| 7. Augment | Optional pre-augmented training set (jitter, scaling, 3-D rotation, magnitude warp) in a **separate** file | train only |
| 8. Save + verify | Arrays, scalers, metadata, data fingerprint, shared `har_common.py`, metrics checklist | — |

**Outputs in `DL Proj/processed/`**

| File | Contents | Used by |
|---|---|---|
| `tabular.npz` | `X_*` (N, 561), `y_*`, `s_*` (subject) for train / val / test | MLP |
| `sequence.npz` | `X_*` (N, 128, 9), `y_*`, `s_*` | CNN, LSTM, Transformer |
| `sequence_train_aug.npz` | training set + 1 augmented copy (optional use) | CNN, LSTM, Transformer |
| `har_common.py` | shared loader, augmentation and **evaluation / results logging** | everyone |
| `meta.json`, `seq_norm.json`, `preprocessors.joblib` | classes, splits, fingerprint, normalization stats | everyone, deployment |
| `METRICS_TO_REPORT.md`, `DATA_CARD.md` | the comparison checklist and data documentation | team / report |
| `figures/` | class balance, example signals, per-class mean signals | report |

Runtime: CPU is enough (≈2–4 min, most of it reading text files from Drive).

## 0 · Configuration and Drive

In [1]:
import os, sys, json, glob, shutil, hashlib, time
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# ---- paths (edit only if your folder names differ) ----
PROJECT_DIR = os.environ.get("HAR_PROJECT_DIR", "/content/drive/MyDrive/DL Proj")
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
PROC_DIR    = os.path.join(PROJECT_DIR, "processed")      # shared output for all models
FIG_DIR     = os.path.join(PROC_DIR, "figures")

# ---- options ----
CLIP_PERCENTILES = (0.01, 99.99)   # outlier clipping on raw signals; set to None to disable
VAL_FOLDS        = 5               # 1/5 of training subjects -> validation (~4 people)
AUG_COPIES       = 1               # augmented copies in sequence_train_aug.npz (0 = don't create)
AUG_OPS          = ("jitter", "scaling", "rotate", "magnitude_warp")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

assert os.path.isdir(PROJECT_DIR), (
    f"Can't find {PROJECT_DIR}. If 'DL Proj' was shared with you, right-click it in Drive -> "
    "Organize -> Add shortcut -> My Drive, then re-run.")
os.makedirs(PROC_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("Project:", PROJECT_DIR, "| Colab:", IN_COLAB)

ValueError: mount failed

## 1 · Locate and validate the raw dataset
The dataset folder is found automatically (it tolerates an extra nesting level or a re-extracted copy). Every expected file is checked before anything is loaded.

In [2]:
hits = [p for p in glob.glob(os.path.join(DATASET_DIR, "**", "activity_labels.txt"), recursive=True)
        if "__MACOSX" not in p]
assert hits, f"activity_labels.txt not found under {DATASET_DIR}"
RAW_ROOT = os.path.dirname(sorted(hits, key=len)[0])
print("Dataset root:", RAW_ROOT)

SIGNALS = ["body_acc_x", "body_acc_y", "body_acc_z",
           "body_gyro_x", "body_gyro_y", "body_gyro_z",
           "total_acc_x", "total_acc_y", "total_acc_z"]
required = ["activity_labels.txt", "features.txt"]
for sp in ("train", "test"):
    required += [f"{sp}/X_{sp}.txt", f"{sp}/y_{sp}.txt", f"{sp}/subject_{sp}.txt"]
    required += [f"{sp}/Inertial Signals/{s}_{sp}.txt" for s in SIGNALS]
missing = [r for r in required if not os.path.isfile(os.path.join(RAW_ROOT, r))]
assert not missing, f"Missing files: {missing}"
print(f"All {len(required)} required files present.")

# Reading ~70 MB of text straight from Drive is slow; copy to the Colab VM first
if IN_COLAB:
    LOCAL = "/content/uci_har_local"
    if not os.path.isdir(LOCAL):
        shutil.copytree(RAW_ROOT, LOCAL, ignore=shutil.ignore_patterns(".DS_Store"))
    RAW_ROOT = LOCAL
    print("Working from local copy:", RAW_ROOT)

## 2 · Load both representations
Row *i* of `X_train.txt` and row *i* of every `Inertial Signals` file describe the same 2.56 s window, so both views stay aligned through every step below.

In [3]:
features = pd.read_csv(f"{RAW_ROOT}/features.txt", sep=r"\s+", header=None)[1].tolist()
act = pd.read_csv(f"{RAW_ROOT}/activity_labels.txt", sep=r"\s+", header=None)
ACTIVITY_MAP = dict(zip(act[0], act[1]))                  # {1: 'WALKING', ...}

raw = {}
for sp in ("train", "test"):
    t0 = time.time()
    X = np.loadtxt(f"{RAW_ROOT}/{sp}/X_{sp}.txt")
    y = np.loadtxt(f"{RAW_ROOT}/{sp}/y_{sp}.txt", dtype=int)
    s = np.loadtxt(f"{RAW_ROOT}/{sp}/subject_{sp}.txt", dtype=int)
    seq = np.stack([np.loadtxt(f"{RAW_ROOT}/{sp}/Inertial Signals/{sig}_{sp}.txt") for sig in SIGNALS], axis=-1)
    raw[sp] = dict(X=X, y=y, s=s, seq=seq)
    print(f"{sp:5s} tabular {X.shape}  sequence {seq.shape}  ({time.time()-t0:.1f}s)")

### Integrity checks
These stop the notebook with a clear message if the files are truncated, mis-aligned or from a different dataset version.

In [4]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond))); assert cond, f"FAILED: {name}"

check("561 feature names", len(features) == 561)
check("6 activity labels", sorted(ACTIVITY_MAP) == [1, 2, 3, 4, 5, 6])
for sp, d in raw.items():
    n = len(d["y"])
    check(f"{sp}: tabular rows == labels", d["X"].shape == (n, 561))
    check(f"{sp}: subjects aligned", d["s"].shape == (n,))
    check(f"{sp}: sequence aligned, 128 steps x 9 channels", d["seq"].shape == (n, 128, 9))
    check(f"{sp}: labels in 1..6", set(np.unique(d["y"])) <= set(range(1, 7)))
    check(f"{sp}: subjects in 1..30", d["s"].min() >= 1 and d["s"].max() <= 30)
official = (len(raw["train"]["y"]), len(raw["test"]["y"])) == (7352, 2947)
checks.append(("official UCI sizes 7352 / 2947", official))
if not official: print("WARNING: split sizes differ from the official UCI release (7352 / 2947)")
check("train/test subjects disjoint", not set(raw["train"]["s"]) & set(raw["test"]["s"]))
print(f"{len(checks)} integrity checks passed")

## 3 · Cleaning

In [5]:
X_all   = np.concatenate([raw["train"]["X"], raw["test"]["X"]]).astype(np.float64)
SEQ_all = np.concatenate([raw["train"]["seq"], raw["test"]["seq"]]).astype(np.float64)
y_raw   = np.concatenate([raw["train"]["y"], raw["test"]["y"]])
subj    = np.concatenate([raw["train"]["s"], raw["test"]["s"]])
origin  = np.array(["train"] * len(raw["train"]["y"]) + ["test"] * len(raw["test"]["y"]))
cleaning = {"rows_in": int(len(y_raw))}

# 3a. features.txt repeats 42 'bandsEnergy' names three times each -> make unique
counts, feat_names = {}, []
for f in features:
    counts[f] = counts.get(f, 0) + 1
    feat_names.append(f if counts[f] == 1 else f"{f}_dup{counts[f]-1}")
cleaning["feature_names_made_unique"] = int(sum(v - 1 for v in counts.values() if v > 1))

# 3b. NaN / inf -> per-column median (tabular) or linear interpolation along time (signals)
X_all[~np.isfinite(X_all)] = np.nan
cleaning["tabular_nonfinite_fixed"] = int(np.isnan(X_all).sum())
if cleaning["tabular_nonfinite_fixed"]:
    med = np.nanmedian(X_all, axis=0); r, c = np.where(np.isnan(X_all)); X_all[r, c] = med[c]
bad = ~np.isfinite(SEQ_all)
cleaning["sequence_nonfinite_fixed"] = int(bad.sum())
if bad.any():
    for i, ch in zip(*np.where(bad.any(axis=1))):
        v = SEQ_all[i, :, ch]; ok = np.isfinite(v)
        SEQ_all[i, :, ch] = np.interp(np.arange(128), np.where(ok)[0], v[ok]) if ok.any() else 0.0

# 3c. exact duplicate windows (same features AND same raw signal) -> drop
key = pd.util.hash_pandas_object(pd.DataFrame(np.concatenate([X_all, SEQ_all.reshape(len(X_all), -1)], 1)), index=False).values
dup = pd.Series(key).duplicated().values
cleaning["duplicate_windows_dropped"] = int(dup.sum())

# 3d. identical windows appearing in two different splits would leak (report; dropped by 3c anyway)
first_origin = pd.Series(origin).groupby(key).transform("first").values
cleaning["cross_split_duplicates"] = int((dup & (first_origin != origin)).sum())

keep = ~dup
X_all, SEQ_all, y_raw, subj, origin = X_all[keep], SEQ_all[keep], y_raw[keep], subj[keep], origin[keep]

# 3e. constant features carry no information
const = np.where(X_all.std(0) == 0)[0]
cleaning["constant_features_dropped"] = [feat_names[i] for i in const]
kept_feat_idx = np.setdiff1d(np.arange(X_all.shape[1]), const)
X_all = X_all[:, kept_feat_idx]; feat_names = [feat_names[i] for i in kept_feat_idx]

# 3f. engineered features are documented as bounded in [-1, 1]
cleaning["tabular_values_outside_[-1,1]"] = int(((X_all < -1.0001) | (X_all > 1.0001)).sum())
cleaning["rows_out"] = int(len(y_raw))
print(json.dumps(cleaning, indent=2))

## 4 · Label encoding
Labels keep the **canonical UCI order** (WALKING = 0 … LAYING = 5) instead of alphabetical, so class ids match the dataset documentation and every teammate's confusion matrix lines up.

In [6]:
CLASSES = [ACTIVITY_MAP[i] for i in range(1, 7)]
y_all = (y_raw - 1).astype(np.int64)
print(dict(enumerate(CLASSES)))

## 5 · Subject-independent train / validation / test split
- **Test** = UCI's official 9 test subjects, so results are comparable with the literature.
- **Validation** = whole people taken from the 21 training subjects, chosen with `StratifiedGroupKFold` and picking the fold whose class mix is closest to the overall mix.
- Windows overlap by 50%, so a random row split would put near-identical windows in train and test. Splitting by person prevents that; the asserts below prove it.

In [7]:
from sklearn.model_selection import StratifiedGroupKFold

trv = np.where(origin == "train")[0]
te  = np.where(origin == "test")[0]
overall = np.bincount(y_all[trv], minlength=6) / len(trv)

best = None
for tr_rel, va_rel in StratifiedGroupKFold(VAL_FOLDS, shuffle=True, random_state=SEED).split(trv, y_all[trv], subj[trv]):
    dist = np.bincount(y_all[trv][va_rel], minlength=6) / len(va_rel)
    score = np.abs(dist - overall).sum()
    if best is None or score < best[0]:
        best = (score, trv[tr_rel], trv[va_rel])
_, tr, va = best
IDX = {"train": tr, "val": va, "test": te}

for a in IDX:
    for b in IDX:
        if a < b:
            assert not set(subj[IDX[a]]) & set(subj[IDX[b]]), f"subject leak {a}/{b}"

split_table = pd.DataFrame({sp: pd.Series(y_all[ix]).map(dict(enumerate(CLASSES))).value_counts() for sp, ix in IDX.items()}).loc[CLASSES]
split_table.loc["TOTAL"] = split_table.sum()
SUBJECTS = {sp: sorted(map(int, np.unique(subj[ix]))) for sp, ix in IDX.items()}
for sp in IDX: print(f"{sp:5s} {len(IDX[sp]):5d} windows  subjects={SUBJECTS[sp]}")
split_table

## 6 · Outlier clipping and normalization (fitted on train only)

In [8]:
from sklearn.preprocessing import StandardScaler
import joblib

seq_tr = SEQ_all[IDX["train"]]
if CLIP_PERCENTILES:
    lo = np.percentile(seq_tr, CLIP_PERCENTILES[0], axis=(0, 1))
    hi = np.percentile(seq_tr, CLIP_PERCENTILES[1], axis=(0, 1))
else:
    lo, hi = np.full(9, -np.inf), np.full(9, np.inf)
clipped = {sp: int(((SEQ_all[ix] < lo) | (SEQ_all[ix] > hi)).sum()) for sp, ix in IDX.items()}
SEQ_c = np.clip(SEQ_all, lo, hi)
cleaning["clipped_values"] = clipped
print("Values clipped per split:", clipped)

seq_mean = SEQ_c[IDX["train"]].mean(axis=(0, 1))
seq_std  = SEQ_c[IDX["train"]].std(axis=(0, 1)) + 1e-8
tab_scaler = StandardScaler().fit(X_all[IDX["train"]])

DATA = {}
for sp, ix in IDX.items():
    DATA[sp] = {
        "tab": tab_scaler.transform(X_all[ix]).astype(np.float32),
        "seq": ((SEQ_c[ix] - seq_mean) / seq_std).astype(np.float32),
        "y": y_all[ix], "s": subj[ix].astype(np.int16)}

print("train seq mean per channel:", DATA["train"]["seq"].mean((0, 1)).round(3))
print("train seq std  per channel:", DATA["train"]["seq"].std((0, 1)).round(3))
print("val   seq mean per channel:", DATA["val"]["seq"].mean((0, 1)).round(3), " (≈0, not exactly — expected)")

## 7 · Exploratory figures (saved for the report)

In [9]:
import matplotlib.pyplot as plt
short = [c.replace("WALKING_", "WALK_") for c in CLASSES]

ax = split_table.drop("TOTAL").set_axis(short).plot.bar(figsize=(9, 3.6), rot=0)
ax.set_title("Windows per activity and split"); ax.set_ylabel("windows"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/class_distribution.png", dpi=130); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 5.5), sharex=True)
for k, ax in enumerate(axes.flat):
    i = np.where(DATA["train"]["y"] == k)[0][0]
    for c in range(3):
        ax.plot(SEQ_c[IDX["train"][i], :, 6 + c], lw=1, label=SIGNALS[6 + c])
    ax.set_title(CLASSES[k], fontsize=10)
axes[0, 0].legend(fontsize=7); fig.supxlabel("timestep (50 Hz, 2.56 s window)"); fig.suptitle("Example raw total acceleration per activity")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/example_signals.png", dpi=130); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for g, ax in enumerate(axes):
    for k in range(6):
        ax.plot(DATA["train"]["seq"][DATA["train"]["y"] == k][:, :, g * 3].mean(0), label=short[k], lw=1.2)
    ax.set_title(f"mean normalized {SIGNALS[g*3]} by class", fontsize=10)
axes[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/class_mean_signals.png", dpi=130); plt.show()

## 8 · Save the shared dataset

In [10]:
np.savez_compressed(f"{PROC_DIR}/tabular.npz", feature_names=np.array(feat_names),
    **{f"{k}_{sp}": DATA[sp][v] for sp in IDX for k, v in [("X", "tab"), ("y", "y"), ("s", "s")]})
np.savez_compressed(f"{PROC_DIR}/sequence.npz", signals=np.array(SIGNALS),
    **{f"{k}_{sp}": DATA[sp][v] for sp in IDX for k, v in [("X", "seq"), ("y", "y"), ("s", "s")]})

# Fingerprint: changes if anything about the processed data changes. Every result row records it.
h = hashlib.sha256()
for sp in ("train", "val", "test"):
    for v in ("tab", "seq", "y"):
        h.update(np.ascontiguousarray(DATA[sp][v]).tobytes())
DATASET_VERSION = "uci-har-" + h.hexdigest()[:12]

seq_norm = {"signals": SIGNALS, "mean": seq_mean.tolist(), "std": seq_std.tolist(),
            "clip_lo": lo.tolist(), "clip_hi": hi.tolist(),
            "how_to_apply": "x = clip(raw, clip_lo, clip_hi); x = (x - mean) / std   # per channel, raw shape (N,128,9)"}
json.dump(seq_norm, open(f"{PROC_DIR}/seq_norm.json", "w"), indent=2)
joblib.dump({"tab_scaler": tab_scaler, "kept_feature_idx": kept_feat_idx, "feature_names": feat_names,
             **seq_norm}, f"{PROC_DIR}/preprocessors.joblib")

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight("balanced", classes=np.arange(6), y=DATA["train"]["y"])

META = {"dataset_version": DATASET_VERSION, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "source": "UCI HAR Dataset (id 240), Anguita et al. 2013, CC BY 4.0",
        "classes": CLASSES, "signals": SIGNALS, "seed": SEED,
        "shapes": {"sequence": [128, 9], "tabular": [len(feat_names)]},
        "split": {"method": "subject-independent; test = official UCI test subjects; "
                            f"val = best-balanced fold of StratifiedGroupKFold({VAL_FOLDS}) over training subjects",
                  "subjects": SUBJECTS, "sizes": {sp: int(len(ix)) for sp, ix in IDX.items()}},
        "class_counts": {sp: np.bincount(DATA[sp]["y"], minlength=6).tolist() for sp in IDX},
        "class_weights": cw.round(6).tolist(),
        "cleaning": cleaning, "clip_percentiles": CLIP_PERCENTILES,
        "normalization": {"tabular": "StandardScaler fit on train", "sequence": "per-channel z-score fit on train (after clipping)"},
        "augmentation": None}
json.dump(META, open(f"{PROC_DIR}/meta.json", "w"), indent=2)
print("DATASET_VERSION =", DATASET_VERSION)

## 9 · Shared module `har_common.py`
Written into the processed folder. Every model notebook imports it for loading, augmentation and — most importantly — **identical metric computation and results logging**.

In [11]:
%%writefile "{PROC_DIR}/har_common.py"
"""
har_common.py  —  shared data loading, augmentation and evaluation
ICT-4442 HAR comparison project (MLP / 1D-CNN / LSTM-BiLSTM / Transformer)

Generated by 01_HAR_Preprocessing.ipynb and saved next to the processed data.
Every model notebook imports THIS file instead of re-implementing loading or
metrics, so all four models are evaluated in exactly the same way.

    import sys; sys.path.insert(0, PROC_DIR)
    import har_common as hc
    data = hc.load_data("sequence")          # (N,128,9) for CNN / LSTM / Transformer
    data = hc.load_data("tabular")           # (N,561)   for the MLP
    ...train your model...
    hc.save_results(model_name="cnn", family="CNN", owner="Akshaj",
                    input_representation="raw_9ch_128",
                    y_test=data["y_test"], p_test=probs_test, s_test=data["s_test"],
                    y_val=data["y_val"],  p_val=probs_val, ...)

Do not edit this file per model. If a change is needed, change it in the
preprocessing notebook and re-run it, so everyone stays on the same version.
"""
import os, json, time, platform, datetime
import numpy as np

PROC_DIR = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(PROC_DIR, "meta.json")) as _f:
    META = json.load(_f)

CLASSES = META["classes"]
N_CLASSES = len(CLASSES)
SIGNALS = META["signals"]
DATASET_VERSION = META["dataset_version"]
RESULTS_DIR = os.path.join(os.path.dirname(PROC_DIR), "results")
SCHEMA_VERSION = "1.0"


# --------------------------------------------------------------------------- #
# Loading
# --------------------------------------------------------------------------- #
def load_data(view="sequence", augmented_train=False):
    """Return a dict with X_/y_/s_ for train, val and test.

    view="sequence": X is (N, 128, 9) float32, per-channel z-scored.
    view="tabular" : X is (N, F) float32, standard-scaled 561-feature vectors.
    y is int64 in 0..5 (order = hc.CLASSES); s is the subject id (1..30).
    augmented_train=True swaps in the pre-augmented training set (sequence only).
    """
    if view not in ("sequence", "tabular"):
        raise ValueError("view must be 'sequence' or 'tabular'")
    with np.load(os.path.join(PROC_DIR, f"{view}.npz")) as d:
        out = {k: d[k] for k in d.files}
    if augmented_train:
        if view != "sequence":
            raise ValueError("Augmentation is only defined for the sequence view.")
        with np.load(os.path.join(PROC_DIR, "sequence_train_aug.npz")) as a:
            out["X_train"], out["y_train"], out["s_train"] = a["X_train"], a["y_train"], a["s_train"]
    return out


def class_weights():
    """Balanced class weights computed on the training split (index = class id)."""
    return np.array(META["class_weights"], dtype=np.float32)


# --------------------------------------------------------------------------- #
# Augmentation (raw-signal space, so rotations stay physically meaningful)
# --------------------------------------------------------------------------- #
with open(os.path.join(PROC_DIR, "seq_norm.json")) as _f:
    SEQ_NORM = json.load(_f)
_MEAN = np.array(SEQ_NORM["mean"], dtype=np.float32)
_STD = np.array(SEQ_NORM["std"], dtype=np.float32)
_TRIPLETS = [(0, 3), (3, 6), (6, 9)]  # body_acc, body_gyro, total_acc xyz


def _jitter(x, rng, sigma=0.05):
    return x + rng.normal(0, sigma, x.shape) * _STD


def _scaling(x, rng, sigma=0.1):
    return x * rng.normal(1, sigma, (x.shape[0], 1, x.shape[2]))


def _time_shift(x, rng, max_shift=10):
    return np.stack([np.roll(xi, rng.integers(-max_shift, max_shift + 1), axis=0) for xi in x])


def _magnitude_warp(x, rng, sigma=0.15, knots=4):
    from scipy.interpolate import CubicSpline
    n, T, C = x.shape
    t, kt = np.arange(T), np.linspace(0, T - 1, knots + 2)
    out = np.empty_like(x)
    for i in range(n):
        curve = np.stack([CubicSpline(kt, rng.normal(1, sigma, knots + 2))(t) for _ in range(C)], axis=1)
        out[i] = x[i] * curve
    return out


def _rotate(x, rng, max_deg=15):
    from scipy.spatial.transform import Rotation
    out = x.copy()
    for i in range(x.shape[0]):
        rot = Rotation.from_euler("xyz", rng.uniform(-max_deg, max_deg, 3), degrees=True).as_matrix()
        for a, b in _TRIPLETS:
            out[i, :, a:b] = x[i, :, a:b] @ rot.T
    return out


AUGMENTERS = {"jitter": _jitter, "scaling": _scaling, "time_shift": _time_shift,
              "magnitude_warp": _magnitude_warp, "rotate": _rotate}


def augment(X, ops=("jitter", "scaling", "rotate", "magnitude_warp"), seed=None):
    """Augment normalized sequences (N,128,9): de-normalize, transform, re-normalize.
    Use only on TRAINING data, e.g. once per epoch for on-the-fly augmentation."""
    rng = np.random.default_rng(seed)
    raw = X * _STD + _MEAN
    for op in ops:
        raw = AUGMENTERS[op](raw, rng)
    return ((raw - _MEAN) / _STD).astype(np.float32)


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def _idx(name):
    return CLASSES.index(name)


def compute_metrics(y_true, prob, subjects=None, has_probabilities=True):
    """All comparison metrics from true labels and class probabilities (N, 6).
    If your model has no probabilities (e.g. plain SVM), pass np.eye(6)[y_pred]
    and has_probabilities=False; log-loss / ROC-AUC / top-2 are then left empty."""
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 precision_recall_fscore_support, confusion_matrix,
                                 cohen_kappa_score, matthews_corrcoef, log_loss,
                                 roc_auc_score, top_k_accuracy_score)
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=np.float64)
    y_pred = prob.argmax(1)
    labels = np.arange(N_CLASSES)
    p, r, f, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": float(f.mean()),
        "macro_precision": float(p.mean()),
        "macro_recall": float(r.mean()),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "log_loss": None, "roc_auc_macro_ovr": None, "top2_accuracy": None,
    }
    if has_probabilities:
        pc = np.clip(prob, 1e-7, 1.0); pc = pc / pc.sum(1, keepdims=True)
        m["log_loss"] = log_loss(y_true, pc, labels=labels)
        m["top2_accuracy"] = top_k_accuracy_score(y_true, pc, k=2, labels=labels)
        try:
            m["roc_auc_macro_ovr"] = roc_auc_score(y_true, pc, multi_class="ovr", average="macro", labels=labels)
        except ValueError:
            pass

    # HAR-specific error analysis (synopsis objective O3)
    sit, stand = _idx("SITTING"), _idx("STANDING")
    walk = [_idx("WALKING"), _idx("WALKING_UPSTAIRS"), _idx("WALKING_DOWNSTAIRS")]
    static = [sit, stand, _idx("LAYING")]
    m["sit_stand_confusion_rate"] = (cm[sit, stand] + cm[stand, sit]) / max(cm[[sit, stand]].sum(), 1)
    wsub = cm[np.ix_(walk, walk)]
    m["walking_variants_confusion_rate"] = (wsub.sum() - np.trace(wsub)) / max(cm[walk].sum(), 1)
    m["static_dynamic_confusion_rate"] = (cm[np.ix_(static, walk)].sum() + cm[np.ix_(walk, static)].sum()) / max(cm.sum(), 1)

    if subjects is not None:
        subjects = np.asarray(subjects)
        per_sub = {int(s): float((y_pred[subjects == s] == y_true[subjects == s]).mean()) for s in np.unique(subjects)}
        vals = np.array(list(per_sub.values()))
        m.update(subject_acc_mean=vals.mean(), subject_acc_std=vals.std(), subject_acc_min=vals.min(),
                 worst_subject=int(min(per_sub, key=per_sub.get)))
        m["per_subject_accuracy"] = per_sub

    per_class = {c: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(sup[i])}
                 for i, c in enumerate(CLASSES)}
    m = {k: (float(v) if isinstance(v, (np.floating, float, np.integer)) and not isinstance(v, bool) else v)
         for k, v in m.items()}
    return m, per_class, cm


def measure_latency(predict_fn, x_one, x_batch=None, n_warmup=10, n_runs=100):
    """Framework-agnostic inference timing. predict_fn(x) must return only after the
    result is ready (e.g. call .cpu() / .numpy() inside it). Returns ms and windows/s."""
    for _ in range(n_warmup):
        predict_fn(x_one)
    t = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); predict_fn(x_one); t.append((time.perf_counter() - t0) * 1000)
    out = {"latency_ms_b1_mean": float(np.mean(t)), "latency_ms_b1_p95": float(np.percentile(t, 95))}
    if x_batch is not None:
        for _ in range(3):
            predict_fn(x_batch)
        t0 = time.perf_counter(); reps = 10
        for _ in range(reps):
            predict_fn(x_batch)
        out["throughput_wps"] = float(reps * len(x_batch) / (time.perf_counter() - t0))
    return out


def hardware_info():
    info = {"platform": platform.platform(), "python": platform.python_version(), "cpu": platform.processor() or platform.machine()}
    try:
        import torch
        info["torch"] = torch.__version__
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    try:
        import tensorflow as tf
        info["tensorflow"] = tf.__version__
        gpus = tf.config.list_physical_devices("GPU")
        if gpus and "gpu" not in info:
            info["gpu"] = str(gpus[0])
    except Exception:
        pass
    return info


# --------------------------------------------------------------------------- #
# The comparison list — every model reports these columns
# --------------------------------------------------------------------------- #
METRIC_DEFINITIONS = [
    # (column, group, description)
    ("model_name", "Identity", "Short id: mlp / cnn / lstm / bilstm / transformer"),
    ("family", "Identity", "MLP, CNN, RNN or Transformer"),
    ("owner", "Identity", "Team member who ran it"),
    ("input_representation", "Identity", "tabular_561 or raw_9ch_128"),
    ("run_tag", "Identity", "baseline / tuned / augmented ..."),
    ("dataset_version", "Identity", "Fingerprint of the processed data; must be identical across all models"),
    ("test_accuracy", "Test performance", "Overall accuracy on the 9 held-out test subjects"),
    ("test_macro_f1", "Test performance", "PRIMARY ranking metric: unweighted mean F1 over the 6 classes"),
    ("test_macro_precision", "Test performance", "Unweighted mean precision over classes"),
    ("test_macro_recall", "Test performance", "Unweighted mean recall over classes"),
    ("test_weighted_f1", "Test performance", "Support-weighted F1"),
    ("test_balanced_accuracy", "Test performance", "Mean per-class recall"),
    ("test_cohen_kappa", "Test performance", "Agreement corrected for chance"),
    ("test_mcc", "Test performance", "Matthews correlation coefficient (multiclass)"),
    ("test_log_loss", "Test performance", "Cross-entropy of predicted probabilities (calibration)"),
    ("test_roc_auc_macro_ovr", "Test performance", "Macro one-vs-rest ROC-AUC"),
    ("test_top2_accuracy", "Test performance", "True class within the top-2 predictions"),
] + [(f"f1_{c}", "Per-class", f"Test F1 for {c}") for c in CLASSES] + [
    ("sit_stand_confusion_rate", "Error analysis", "Share of SITTING/STANDING windows predicted as the other one"),
    ("walking_variants_confusion_rate", "Error analysis", "Share of walking windows confused among walk / upstairs / downstairs"),
    ("static_dynamic_confusion_rate", "Error analysis", "Share of all windows where a static activity was confused with a moving one or vice versa"),
    ("subject_acc_mean", "Robustness", "Mean of per-test-subject accuracy"),
    ("subject_acc_std", "Robustness", "Spread of accuracy across test subjects (lower = more consistent)"),
    ("subject_acc_min", "Robustness", "Accuracy on the worst test subject"),
    ("val_accuracy", "Generalization", "Accuracy on the validation subjects"),
    ("val_macro_f1", "Generalization", "Macro-F1 on validation (used for model selection / early stopping)"),
    ("train_macro_f1", "Generalization", "Macro-F1 on the training set"),
    ("gap_train_test_f1", "Generalization", "train_macro_f1 - test_macro_f1 (overfitting indicator)"),
    ("params_total", "Efficiency", "Total number of parameters"),
    ("model_size_mb", "Efficiency", "Size on disk of the saved weights file"),
    ("flops_per_window", "Efficiency", "FLOPs for one 2.56 s window (state the tool used in notes)"),
    ("latency_ms_cpu_b1", "Efficiency", "Mean CPU inference time for ONE window, batch size 1 (deployment proxy)"),
    ("latency_ms_gpu_b1", "Efficiency", "Mean GPU inference time for one window"),
    ("throughput_gpu_wps", "Efficiency", "Windows/second on GPU at batch 256"),
    ("peak_gpu_mem_mb", "Efficiency", "Peak GPU memory during training"),
    ("train_time_s", "Training cost", "Wall-clock training time"),
    ("epochs_trained", "Training cost", "Epochs actually run (after early stopping)"),
    ("best_epoch", "Training cost", "Epoch with best val macro-F1"),
    ("time_per_epoch_s", "Training cost", "Mean seconds per epoch"),
    ("hardware", "Reproducibility", "GPU / CPU used"),
    ("framework", "Reproducibility", "e.g. PyTorch 2.x, TensorFlow 2.x, scikit-learn"),
    ("seed", "Reproducibility", "Random seed"),
    ("notes", "Reproducibility", "Anything a reader needs to interpret the row"),
]
COMPARISON_COLUMNS = ["run_id", "timestamp"] + [c for c, _, _ in METRIC_DEFINITIONS]


def save_results(model_name, family, owner, input_representation, y_test, p_test,
                 s_test=None, y_val=None, p_val=None, y_train=None, p_train=None,
                 efficiency=None, training=None, hyperparameters=None, framework="",
                 artifacts=None, notes="", run_tag="baseline", seed=None,
                 has_probabilities=True, history=None):
    """Compute every comparison metric, write the full record + plots to
    results/<model_name>/<run_tag>/, and append one row to results/comparison_table.csv.
    Returns the record dict."""
    import pandas as pd
    efficiency, training = efficiency or {}, training or {}
    ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    run_id = f"{model_name}__{run_tag}__{ts}"
    out_dir = os.path.join(RESULTS_DIR, model_name, run_tag)
    os.makedirs(out_dir, exist_ok=True)

    test_m, test_pc, cm = compute_metrics(y_test, p_test, s_test, has_probabilities)
    val_m = compute_metrics(y_val, p_val, None, has_probabilities)[0] if p_val is not None else {}
    train_m = compute_metrics(y_train, p_train, None, has_probabilities)[0] if p_train is not None else {}
    hw = hardware_info()

    record = {
        "schema_version": SCHEMA_VERSION, "run_id": run_id, "timestamp": ts,
        "model_name": model_name, "family": family, "owner": owner,
        "input_representation": input_representation, "run_tag": run_tag,
        "dataset_version": DATASET_VERSION, "framework": framework, "seed": seed,
        "hardware": hw, "test": test_m, "test_per_class": test_pc,
        "confusion_matrix": cm.tolist(), "classes": CLASSES,
        "val": val_m, "train": train_m, "efficiency": efficiency, "training": training,
        "hyperparameters": hyperparameters or {}, "artifacts": artifacts or {}, "notes": notes,
    }
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(record, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    if history is not None:
        pd.DataFrame(history).to_csv(os.path.join(out_dir, "history.csv"), index=False)

    from sklearn.metrics import classification_report
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(classification_report(np.asarray(y_test), np.asarray(p_test).argmax(1),
                                      labels=list(range(N_CLASSES)), target_names=CLASSES, digits=4, zero_division=0))
    import matplotlib.pyplot as plt
    plt.close(plot_confusion(cm, os.path.join(out_dir, "confusion_matrix.png"), title=f"{model_name} ({run_tag}) — test"))

    row = {"run_id": run_id, "timestamp": ts, "model_name": model_name, "family": family, "owner": owner,
           "input_representation": input_representation, "run_tag": run_tag,
           "dataset_version": DATASET_VERSION, "framework": framework, "seed": seed, "notes": notes,
           "hardware": hw.get("gpu", hw.get("cpu"))}
    for k in ["accuracy", "macro_f1", "macro_precision", "macro_recall", "weighted_f1", "balanced_accuracy",
              "cohen_kappa", "mcc", "log_loss", "roc_auc_macro_ovr", "top2_accuracy"]:
        row[f"test_{k}"] = test_m.get(k)
    for c in CLASSES:
        row[f"f1_{c}"] = test_pc[c]["f1"]
    for k in ["sit_stand_confusion_rate", "walking_variants_confusion_rate", "static_dynamic_confusion_rate",
              "subject_acc_mean", "subject_acc_std", "subject_acc_min"]:
        row[k] = test_m.get(k)
    row["val_accuracy"], row["val_macro_f1"] = val_m.get("accuracy"), val_m.get("macro_f1")
    row["train_macro_f1"] = train_m.get("macro_f1")
    row["gap_train_test_f1"] = (train_m["macro_f1"] - test_m["macro_f1"]) if train_m else None
    for k in ["params_total", "model_size_mb", "flops_per_window", "latency_ms_cpu_b1", "latency_ms_gpu_b1",
              "throughput_gpu_wps", "peak_gpu_mem_mb"]:
        row[k] = efficiency.get(k, training.get(k))
    for k in ["train_time_s", "epochs_trained", "best_epoch", "time_per_epoch_s"]:
        row[k] = training.get(k)

    table = os.path.join(RESULTS_DIR, "comparison_table.csv")
    new = pd.DataFrame([row], columns=COMPARISON_COLUMNS)
    if os.path.exists(table):
        new = pd.concat([pd.read_csv(table), new], ignore_index=True)
    new.to_csv(table, index=False)
    print(f"Saved {run_id}\n  details -> {out_dir}\n  table   -> {table}")
    return record


def plot_confusion(cm, path=None, title="Confusion matrix"):
    import matplotlib.pyplot as plt
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    short = [c.replace("WALKING_", "WALK_") for c in CLASSES]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, mat, sub, fmt in [(axes[0], cm, "counts", "d"), (axes[1], cmn, "row-normalized (recall)", ".2f")]:
        im = ax.imshow(mat, cmap="Blues")
        ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(short, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(short, fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(sub, fontsize=10)
        thr = mat.max() / 2
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="white" if mat[i, j] > thr else "black")
    fig.suptitle(title); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    return fig


def write_metrics_guide(path):
    """Write METRICS_TO_REPORT.md — the checklist to share with teammates."""
    lines = ["# Metrics every model must report", "",
             f"Dataset version: `{DATASET_VERSION}` (check that your run shows the same value).", "",
             "Use `har_common.save_results(...)` and all of these are filled in automatically, "
             "except the efficiency / training-cost numbers, which you pass in.", ""]
    group = None
    for col, g, desc in METRIC_DEFINITIONS:
        if g != group:
            lines += ["", f"## {g}", "", "| Column | Meaning |", "|---|---|"]
            group = g
        lines.append(f"| `{col}` | {desc} |")
    lines += ["", "## Protocol rules", "",
              "1. Use the processed data as saved. Never refit scalers or re-split.",
              "2. Select hyperparameters / early-stop on **validation macro-F1**. Evaluate on test once, at the end.",
              "3. Seed 42. Same test subjects for everyone (UCI official test split).",
              "4. Measure CPU latency with batch size 1, after warm-up (`hc.measure_latency`).",
              "5. Baseline runs use `run_tag='baseline'`; tuned runs use `run_tag='tuned'`."]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    return path

## 10 · Augmented training set (optional for model owners)
Augmentation is done in **raw sensor units** (de-normalize → transform → re-normalize), so the 3-D rotation rotates real x/y/z axes. It is saved in a separate file: baseline runs use clean data; anyone comparing *with augmentation* loads `hc.load_data('sequence', augmented_train=True)`, and since everyone gets the same augmented windows, the comparison stays fair.

In [12]:
sys.path.insert(0, PROC_DIR)
import importlib, har_common as hc
importlib.reload(hc)

if AUG_COPIES > 0:
    Xa, ya, sa = [DATA["train"]["seq"]], [DATA["train"]["y"]], [DATA["train"]["s"]]
    for c in range(AUG_COPIES):
        Xa.append(hc.augment(DATA["train"]["seq"], ops=AUG_OPS, seed=SEED + c))
        ya.append(DATA["train"]["y"]); sa.append(DATA["train"]["s"])
    np.savez_compressed(f"{PROC_DIR}/sequence_train_aug.npz", X_train=np.concatenate(Xa),
                        y_train=np.concatenate(ya), s_train=np.concatenate(sa))
    META["augmentation"] = {"file": "sequence_train_aug.npz", "copies": AUG_COPIES, "ops": list(AUG_OPS),
                            "train_windows": int(sum(len(x) for x in Xa))}
    json.dump(META, open(f"{PROC_DIR}/meta.json", "w"), indent=2)
    print("Augmented train:", DATA["train"]["seq"].shape, "->", np.concatenate(Xa).shape)

    fig, ax = plt.subplots(1, 2, figsize=(13, 3), sharey=True)
    for a, X_, t in [(ax[0], Xa[0][0], "original"), (ax[1], Xa[1][0], "augmented")]:
        for c in range(3): a.plot(X_[:, c], lw=1, label=SIGNALS[c])
        a.set_title(f"{CLASSES[DATA['train']['y'][0]]} — {t}")
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/augmentation_example.png", dpi=130); plt.show()

## 11 · Metrics checklist and data card

In [13]:
importlib.reload(hc)
hc.write_metrics_guide(f"{PROC_DIR}/METRICS_TO_REPORT.md")

card = f'''# Processed UCI HAR — data card
Version: `{DATASET_VERSION}`  ·  created {META["created"]}

## Splits (subject-independent)
| split | windows | subjects |
|---|---|---|
''' + "\n".join(f"| {sp} | {META['split']['sizes'][sp]} | {', '.join(map(str, SUBJECTS[sp]))} |" for sp in IDX) + f'''

## Classes (id: name)
{", ".join(f"{i}: {c}" for i, c in enumerate(CLASSES))}

## Representations
- `sequence.npz` — X (N, 128, 9) float32; channels: {", ".join(SIGNALS)}; clipped at train percentiles {CLIP_PERCENTILES}, then per-channel z-scored with train statistics.
- `tabular.npz` — X (N, {len(feat_names)}) float32; 561 UCI engineered features, StandardScaler fit on train.
- Keys: X_train, y_train, s_train (subject), X_val, ..., X_test, ...

## Cleaning log
```
{json.dumps(cleaning, indent=2)}
```

## Usage
```python
import sys; sys.path.insert(0, "{PROC_DIR}")
import har_common as hc
data = hc.load_data("sequence")   # or "tabular"
```
'''
open(f"{PROC_DIR}/DATA_CARD.md", "w").write(card)
print(open(f"{PROC_DIR}/METRICS_TO_REPORT.md").read()[:1500], "...")

## 12 · Final verification
Reloads the saved files exactly the way a teammate would, re-checks them, and runs a 20-second logistic-regression sanity baseline. On the real data this should give **~95% test accuracy**; much lower means something upstream is wrong.

In [14]:
importlib.reload(hc)
for view in ("sequence", "tabular"):
    d = hc.load_data(view)
    for sp in ("train", "val", "test"):
        X, y = d[f"X_{sp}"], d[f"y_{sp}"]
        assert np.isfinite(X).all() and X.dtype == np.float32 and len(X) == len(y) == len(d[f"s_{sp}"])
        assert set(np.unique(y)) == set(range(6)), f"{view}/{sp} is missing a class"
    print(f"{view:8s} OK  train {d['X_train'].shape}  val {d['X_val'].shape}  test {d['X_test'].shape}")
assert hc.DATASET_VERSION == DATASET_VERSION

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
d = hc.load_data("tabular")
lr = LogisticRegression(max_iter=1000).fit(d["X_train"], d["y_train"])
p = lr.predict(d["X_test"])
print(f"\nSanity baseline (logistic regression, tabular): test acc {accuracy_score(d['y_test'], p):.4f}  macro-F1 {f1_score(d['y_test'], p, average='macro'):.4f}")
print("\nFiles in", PROC_DIR)
for f in sorted(os.listdir(PROC_DIR)):
    if os.path.isfile(os.path.join(PROC_DIR, f)):
        print(f"   {f:28s} {os.path.getsize(os.path.join(PROC_DIR, f))/1e6:6.2f} MB")

## For teammates
Open your own model notebook in Colab, mount Drive, and start with:
```python
import sys; sys.path.insert(0, "/content/drive/MyDrive/DL Proj/processed")
import har_common as hc
data = hc.load_data("tabular")     # MLP
data = hc.load_data("sequence")    # CNN / LSTM / Transformer  -> X (N, 128, 9)
```
Train, then call `hc.save_results(...)` with test/val/train probabilities and your efficiency numbers. That writes your full metrics to `DL Proj/results/<model>/` and adds your row to `DL Proj/results/comparison_table.csv`. See `02_Transformer_Baseline.ipynb` for a complete example and `METRICS_TO_REPORT.md` for the list.